# Stage 4 — Phase 4 Zero-Shot: Molmo2-O-7B Only

Mounts Drive once, just to copy the 20 test sheets (images + labels) into local
`/content` storage — then everything else (tiling, inference, scoring) runs off the
fast local copy, not repeated slow Drive-mounted I/O.

Runs Molmo2 zero-shot over the real 20 frozen Gupta test sheets (tiled per checklist
2.3), stitches+dedups per-tile predictions to sheet level (3.3), and scores real
detection precision/recall/F1 against ground truth (3.1/3.2 — point-in-GT-box matching).

## 1. Mount Drive, copy the 20 test sheets to local storage

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/pid_project/data")
raw = DRIVE_ROOT / "gupta_pid" / "PID_Dataset" / "0__raw_data"

SHEETS_TEST_DIR = Path("/content/gupta_test/sheets_test")
LABELS_TEST_DIR = Path("/content/gupta_test/labels_test")
SHEETS_TEST_DIR.mkdir(parents=True, exist_ok=True)
LABELS_TEST_DIR.mkdir(parents=True, exist_ok=True)

for src_dir, dst_dir in [(raw / "sheets" / "test", SHEETS_TEST_DIR), (raw / "labels" / "test", LABELS_TEST_DIR)]:
    for f in src_dir.iterdir():
        shutil.copy(f, dst_dir / f.name)

print("sheets_test:", len(list(SHEETS_TEST_DIR.glob("*"))), "files")
print("labels_test:", len(list(LABELS_TEST_DIR.glob("*"))), "files")
print("\nCopy done — everything below runs off local /content storage, not Drive.")

Mounted at /content/drive
sheets_test: 20 files
labels_test: 20 files

Copy done — everything below runs off local /content storage, not Drive.


## 2. Check GPU before installing anything

Run this first — if it errors or shows nothing, the runtime has no GPU (change
Runtime type before continuing, per the earlier CPU-runtime incident).

In [3]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

NVIDIA A100-SXM4-80GB, 81920 MiB


## 3. Install dependencies

Molmo2-O-7B requires `transformers==4.57.1` exactly (its own model card pins this).
**Deliberately NOT installing `torch` generically** — Colab's preinstalled CUDA build
gets silently replaced by a CPU-only PyPI wheel if you do (bit us twice already this
session).

In [4]:
!pip install -q transformers==4.57.1 accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 137.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


## ⚠️ Restart the runtime now (Runtime → Restart session), then continue from below

## 4. Config + shared imports

In [5]:
import json, re, time
from pathlib import Path
import torch
from PIL import Image, ImageDraw
from transformers import (
    AutoModelForImageTextToText, AutoProcessor,
    MaxTimeCriteria, StoppingCriteriaList,
)

SHEETS_TEST_DIR = Path("/content/gupta_test/sheets_test")
LABELS_TEST_DIR = Path("/content/gupta_test/labels_test")
print("Sheets:", SHEETS_TEST_DIR.exists(), "| Labels:", LABELS_TEST_DIR.exists())
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU — check Runtime type before continuing"

Sheets: True | Labels: True
torch: 2.11.0+cu128 | CUDA available: True


## 5. Tiling (checklist 2.3) + scoring/stitch harness (checklist 3.1-3.3) — inline

In [6]:
TILE_SIZE, OVERLAP = 1024, 205

def compute_tile_grid(img_w, img_h, tile_size=TILE_SIZE, overlap=OVERLAP):
    stride = tile_size - overlap
    tiles = []
    y0 = 0
    while y0 < img_h:
        y1 = min(y0 + tile_size, img_h)
        x0 = 0
        while x0 < img_w:
            x1 = min(x0 + tile_size, img_w)
            tiles.append({"x0": x0, "y0": y0, "x1": x1, "y1": y1})
            x0 += stride
        y0 += stride
    return tiles

def yolo_line_to_xyxy(line, img_w, img_h):
    parts = line.split()
    cx, cy, w, h = (float(v) for v in parts[1:5])
    cx, cy, w, h = cx * img_w, cy * img_h, w * img_w, h * img_h
    return [cx - w/2, cy - h/2, cx + w/2, cy + h/2]

def load_gt_boxes(label_path, img_w, img_h):
    """Absolute sheet-coordinate GT boxes (class-agnostic — Gupta rule 5)."""
    if not label_path.exists():
        return []
    return [yolo_line_to_xyxy(l, img_w, img_h) for l in label_path.read_text().splitlines() if l.strip()]

# --- scoring.py (checklist 3.1/3.2) ---
def _bbox_center(bbox):
    x0, y0, x1, y1 = bbox
    return (x0 + x1) / 2, (y0 + y1) / 2

def point_in_box(point, box):
    x, y = point
    x0, y0, x1, y1 = box
    return x0 <= x <= x1 and y0 <= y <= y1

def prediction_point(pred):
    if pred.get("point") is not None:
        return tuple(pred["point"])
    return _bbox_center(pred["bbox"])

def match_predictions_to_gt(predictions, gt_boxes):
    order = sorted(range(len(predictions)),
                   key=lambda i: (predictions[i].get("confidence") is None, -(predictions[i].get("confidence") or 0)))
    gt_taken = [False] * len(gt_boxes)
    n_matched = 0
    for i in order:
        pt = prediction_point(predictions[i])
        for j, box in enumerate(gt_boxes):
            if not gt_taken[j] and point_in_box(pt, box):
                gt_taken[j] = True
                n_matched += 1
                break
    return n_matched, len(predictions), len(gt_boxes)

def precision_recall_f1(predictions, gt_boxes):
    n_matched, n_pred, n_gt = match_predictions_to_gt(predictions, gt_boxes)
    precision = n_matched / n_pred if n_pred else (1.0 if n_gt == 0 else 0.0)
    recall = n_matched / n_gt if n_gt else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "n_matched": n_matched, "n_pred": n_pred, "n_gt": n_gt}

# --- stitch.py (checklist 3.3) ---
DEDUP_DISTANCE_PX = 30

def remap_tile_to_sheet(pred, tile_origin):
    ox, oy = tile_origin
    out = dict(pred)
    x0, y0, x1, y1 = pred["bbox"]
    out["bbox"] = [x0 + ox, y0 + oy, x1 + ox, y1 + oy]
    if pred.get("point") is not None:
        px, py = pred["point"]
        out["point"] = [px + ox, py + oy]
    return out

def stitch_and_dedup(tile_predictions, tile_origins, dedup_distance_px=DEDUP_DISTANCE_PX):
    sheet_preds = []
    for preds, origin in zip(tile_predictions, tile_origins):
        for p in preds:
            sheet_preds.append(remap_tile_to_sheet(p, origin))
    clusters = []
    for pred in sheet_preds:
        pt = prediction_point(pred)
        etype = pred.get("entity_type")
        placed = False
        for cluster in clusters:
            if cluster["type"] != etype:
                continue
            cx, cy = prediction_point(cluster["members"][0])
            if ((pt[0]-cx)**2 + (pt[1]-cy)**2)**0.5 <= dedup_distance_px:
                cluster["members"].append(pred)
                placed = True
                break
        if not placed:
            clusters.append({"type": etype, "members": [pred]})
    deduped = []
    for cluster in clusters:
        members = cluster["members"]
        with_conf = [m for m in members if m.get("confidence") is not None]
        best = max(with_conf, key=lambda m: m["confidence"]) if with_conf else members[0]
        deduped.append(best)
    return deduped, len(sheet_preds)

## 6. Molmo2-O-7B: load, prompt, parse

In [ ]:
MOLMO_MODEL_ID = "allenai/Molmo2-O-7B"

# v1 (unengineered, used in the earlier zero-shot run — kept for reference/re-comparison)
MOLMO_PROMPT_V1 = (
    "Point to every symbol (valve, instrument, flange, nozzle, safety device, or other "
    "P&ID symbol) visible in this image."
)

# v2 (engineered) — informed by the real v1 results: strong density-degradation (both
# Molmo2 and Claude got worse on dense sheets) and mediocre precision (both models).
# Changes: (1) explicit exhaustive/systematic-scan instruction, since dense sheets are
# exactly where both models previously fell apart; (2) explicit symbol definition matching
# Gupta ground truth (rule 5 - class-agnostic icons only, not lines/text/dimensions),
# aimed at the shared precision problem; (3) phrased in Molmo's native "point to X"
# convention rather than a generic "detect" framing.
MOLMO_PROMPT_V2 = (
    "Point to every distinct P&ID symbol in this image - valves, instruments (circular "
    "gauges/bubbles), flanges, nozzles, safety devices, reducers, and similar equipment "
    "icons. This tile may contain many symbols, sometimes 50 or more - scan the entire "
    "image carefully from top-left to bottom-right and point to all of them, not just the "
    "most obvious ones. Do not point to plain pipe or line segments, text labels alone, or "
    "dimension arrows - only actual symbol icons."
)

# v3 (engineered, shorter) — v2 was too long/paragraph-y for a pointing model trained on
# short imperative instructions; the negative "do not point to..." clauses are also poorly
# handled by small pointing models. v3 returns to Molmo's native short "point to all X"
# phrasing, keeps the exhaustiveness nudge ("including small ones"), drops negatives.
MOLMO_PROMPT_V3 = (
    "Point to all P&ID equipment symbols: valves, instrument circles, flanges, "
    "nozzles, safety devices, and reducers. Point to each one individually, "
    "including small ones."
)

MOLMO_PROMPT = MOLMO_PROMPT_V3  # active prompt used by run_molmo() below

_POINTS_RE = re.compile(r'<(?:points|tracks).*? coords="([0-9\t:;, .]+)"/?>')

def load_molmo():
    t0 = time.time()
    processor = AutoProcessor.from_pretrained(MOLMO_MODEL_ID, trust_remote_code=True, dtype="auto")
    model = AutoModelForImageTextToText.from_pretrained(
        MOLMO_MODEL_ID, trust_remote_code=True, dtype="auto", device_map="cuda"
    )
    print(f"Loaded {MOLMO_MODEL_ID} in {time.time()-t0:.1f}s")
    print("VRAM used:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")
    return processor, model

def run_molmo(processor, model, image, prompt=MOLMO_PROMPT, max_time_s=60.0):
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}, {"type": "image", "image": image}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
    ).to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=2048, do_sample=False,
            stopping_criteria=StoppingCriteriaList([MaxTimeCriteria(max_time=max_time_s)]),
        )
    latency = time.time() - t0
    gen_tokens = out[:, inputs["input_ids"].shape[1]:]
    text = processor.batch_decode(gen_tokens, skip_special_tokens=True)[0]
    return text, latency

def parse_molmo(text, img_w, img_h):
    detections = []
    for m in _POINTS_RE.finditer(text):
        nums = [float(v) for v in re.split(r'[\t:;, ]+', m.group(1).strip()) if v]
        if len(nums) % 3 != 0:
            if len(nums) >= 2 and nums[0] == nums[1] and (len(nums)-1) % 3 == 0:
                nums = nums[1:]
            else:
                return None, f"coords not a multiple of 3: {nums}"
        for i in range(0, len(nums), 3):
            _frame, x_scaled, y_scaled = nums[i:i+3]
            x, y = x_scaled/1000*img_w, y_scaled/1000*img_h
            detections.append({"bbox": [x-20, y-20, x+20, y+20], "confidence": None, "entity_type": None, "point": [x, y]})
    if not detections and "<point" in text.lower():
        return None, "contains point-like tags but regex found no matches"
    return detections, None

## 7. Load model, run zero-shot over all 20 frozen test sheets, score

Per checklist 2.6: ~127 tiles total across 20 sheets, ~4.4s/tile measured earlier for
Molmo2 zero-shot → expect roughly 10 minutes.

In [8]:
processor, model = load_molmo()

test_ids = sorted(p.stem for p in SHEETS_TEST_DIR.glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
print(f"Found {len(test_ids)} test sheets (expected 20)")

sheet_results = []
totals = {"n_matched": 0, "n_pred": 0, "n_gt": 0}
parse_failures = 0
total_tiles = 0

for sheet_id in test_ids:
    img_path = next(SHEETS_TEST_DIR.glob(f"{sheet_id}.*"))
    label_path = LABELS_TEST_DIR / f"{sheet_id}.txt"
    sheet_img = Image.open(img_path).convert("RGB")
    W, H = sheet_img.size
    gt_boxes = load_gt_boxes(label_path, W, H)

    tiles = compute_tile_grid(W, H)
    tile_preds, tile_origins = [], []
    for t in tiles:
        total_tiles += 1
        crop = sheet_img.crop((t["x0"], t["y0"], t["x1"], t["y1"]))
        raw_text, _latency = run_molmo(processor, model, crop)
        dets, err = parse_molmo(raw_text, t["x1"]-t["x0"], t["y1"]-t["y0"])
        if err:
            parse_failures += 1
            dets = []
        tile_preds.append(dets)
        tile_origins.append((t["x0"], t["y0"]))

    sheet_preds, n_raw = stitch_and_dedup(tile_preds, tile_origins)
    result = precision_recall_f1(sheet_preds, gt_boxes)
    sheet_results.append({"sheet_id": sheet_id, **result})
    for k in totals:
        totals[k] += result[k]
    print(f"{sheet_id:10s} tiles={len(tiles):2d} gt={result['n_gt']:3d} pred={result['n_pred']:3d} "
          f"matched={result['n_matched']:3d} precision={result['precision']:.2f} recall={result['recall']:.2f} f1={result['f1']:.2f}")

print(f"\n--- Overall (micro-averaged across all {len(test_ids)} test sheets, {total_tiles} tiles) ---")
overall_precision = totals["n_matched"] / totals["n_pred"] if totals["n_pred"] else 0.0
overall_recall = totals["n_matched"] / totals["n_gt"] if totals["n_gt"] else 0.0
overall_f1 = 2*overall_precision*overall_recall/(overall_precision+overall_recall) if (overall_precision+overall_recall) else 0.0
print(f"Precision: {overall_precision:.3f} | Recall: {overall_recall:.3f} | F1: {overall_f1:.3f}")
print(f"Parse failures: {parse_failures}/{total_tiles} tiles ({100*parse_failures/total_tiles:.1f}%)")
print(f"\nProvisional pass bar (Stage4_Checklist_Status.md 3.6): F1 >= 0.70")
print("✓ MEETS bar" if overall_f1 >= 0.70 else "✗ below bar")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

processing_molmo2.py: 0.00B [00:00, ?B/s]

image_processing_molmo2.py: 0.00B [00:00, ?B/s]

...

model-00006-of-00007.safetensors:   0%|          | 0.00/4.86G [00:00<?, ?B/s]

model-00004-of-00007.safetensors:   0%|          | 0.00/4.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

Loaded allenai/Molmo2-O-7B in 104.6s
VRAM used: 31.0 GB
Found 20 test sheets (expected 20)
0          tiles= 2 gt= 11 pred= 11 matched= 11 precision=1.00 recall=1.00 f1=1.00
103        tiles= 2 gt= 18 pred= 16 matched= 15 precision=0.94 recall=0.83 f1=0.88
11         tiles= 1 gt= 49 pred= 17 matched= 15 precision=0.88 recall=0.31 f1=0.45
124        tiles= 1 gt=  8 pred=  8 matched=  8 precision=1.00 recall=1.00 f1=1.00
129        tiles= 1 gt= 22 pred= 22 matched= 22 precision=1.00 recall=1.00 f1=1.00
136        tiles= 4 gt=  9 pred= 17 matched=  9 precision=0.53 recall=1.00 f1=0.69
145        tiles= 4 gt=  2 pred=  6 matched=  1 precision=0.17 recall=0.50 f1=0.25
148        tiles= 6 gt= 11 pred=  6 matched=  3 precision=0.50 recall=0.27 f1=0.35
15         tiles= 2 gt= 44 pred= 12 matched= 11 precision=0.92 recall=0.25 f1=0.39
151        tiles=12 gt=101 pred=117 matched= 46 precision=0.39 recall=0.46 f1=0.42
157        tiles= 6 gt= 46 pred=  6 matched=  6 precision=1.00 recall=0.13 f1=0

---
## 8. Claude (claude-sonnet-4-6) — same test, same data, same harness

This is the optional incumbent-comparison column (checklist 3.4), run for real: the
cloud model the agent currently uses, scored with the exact same tiling + point-in-box
matching + stitch/dedup harness as Molmo2, on the same 20 frozen test sheets. Reference
only — per CLAUDE.md, ground truth is the pass bar, not this comparison.

Self-contained (works even if you didn't run the Molmo section above/this session).
No GPU needed — this is an API call, not a local model.

## 8a. Install SDK + config

In [9]:
!pip install -q anthropic

import os
os.environ["ANTHROPIC_API_KEY"] = "paste-your-anthropic-api-key-here"  # <-- edit this each session; never commit a real key (GitHub blocks it anyway)

CLAUDE_MODEL = "claude-sonnet-4-6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 30.7 MB/s eta 0:00:00


## 8b. Tiling + scoring/stitch harness (same as section 5 — duplicated here for standalone use)

In [10]:
import json, re, time
from pathlib import Path
from PIL import Image

TILE_SIZE, OVERLAP = 1024, 205

def compute_tile_grid(img_w, img_h, tile_size=TILE_SIZE, overlap=OVERLAP):
    stride = tile_size - overlap
    tiles = []
    y0 = 0
    while y0 < img_h:
        y1 = min(y0 + tile_size, img_h)
        x0 = 0
        while x0 < img_w:
            x1 = min(x0 + tile_size, img_w)
            tiles.append({"x0": x0, "y0": y0, "x1": x1, "y1": y1})
            x0 += stride
        y0 += stride
    return tiles

def yolo_line_to_xyxy(line, img_w, img_h):
    parts = line.split()
    cx, cy, w, h = (float(v) for v in parts[1:5])
    cx, cy, w, h = cx * img_w, cy * img_h, w * img_w, h * img_h
    return [cx - w/2, cy - h/2, cx + w/2, cy + h/2]

def load_gt_boxes(label_path, img_w, img_h):
    if not label_path.exists():
        return []
    return [yolo_line_to_xyxy(l, img_w, img_h) for l in label_path.read_text().splitlines() if l.strip()]

def _bbox_center(bbox):
    x0, y0, x1, y1 = bbox
    return (x0 + x1) / 2, (y0 + y1) / 2

def point_in_box(point, box):
    x, y = point
    x0, y0, x1, y1 = box
    return x0 <= x <= x1 and y0 <= y <= y1

def prediction_point(pred):
    if pred.get("point") is not None:
        return tuple(pred["point"])
    return _bbox_center(pred["bbox"])

def match_predictions_to_gt(predictions, gt_boxes):
    order = sorted(range(len(predictions)),
                   key=lambda i: (predictions[i].get("confidence") is None, -(predictions[i].get("confidence") or 0)))
    gt_taken = [False] * len(gt_boxes)
    n_matched = 0
    for i in order:
        pt = prediction_point(predictions[i])
        for j, box in enumerate(gt_boxes):
            if not gt_taken[j] and point_in_box(pt, box):
                gt_taken[j] = True
                n_matched += 1
                break
    return n_matched, len(predictions), len(gt_boxes)

def precision_recall_f1(predictions, gt_boxes):
    n_matched, n_pred, n_gt = match_predictions_to_gt(predictions, gt_boxes)
    precision = n_matched / n_pred if n_pred else (1.0 if n_gt == 0 else 0.0)
    recall = n_matched / n_gt if n_gt else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "n_matched": n_matched, "n_pred": n_pred, "n_gt": n_gt}

DEDUP_DISTANCE_PX = 30

def remap_tile_to_sheet(pred, tile_origin):
    ox, oy = tile_origin
    out = dict(pred)
    x0, y0, x1, y1 = pred["bbox"]
    out["bbox"] = [x0 + ox, y0 + oy, x1 + ox, y1 + oy]
    return out

def stitch_and_dedup(tile_predictions, tile_origins, dedup_distance_px=DEDUP_DISTANCE_PX):
    sheet_preds = []
    for preds, origin in zip(tile_predictions, tile_origins):
        for p in preds:
            sheet_preds.append(remap_tile_to_sheet(p, origin))
    clusters = []
    for pred in sheet_preds:
        pt = prediction_point(pred)
        etype = pred.get("entity_type")
        placed = False
        for cluster in clusters:
            if cluster["type"] != etype:
                continue
            cx, cy = prediction_point(cluster["members"][0])
            if ((pt[0]-cx)**2 + (pt[1]-cy)**2)**0.5 <= dedup_distance_px:
                cluster["members"].append(pred)
                placed = True
                break
        if not placed:
            clusters.append({"type": etype, "members": [pred]})
    deduped = []
    for cluster in clusters:
        members = cluster["members"]
        with_conf = [m for m in members if m.get("confidence") is not None]
        best = max(with_conf, key=lambda m: m["confidence"]) if with_conf else members[0]
        deduped.append(best)
    return deduped, len(sheet_preds)

## 8c. Claude prompt, call, parse (reuses the same box-JSON format as Qwen/InternVL)

In [ ]:
import base64
from io import BytesIO
import anthropic

claude_client = anthropic.Anthropic()

# v1 (unengineered, used in the earlier zero-shot run — kept for reference/re-comparison)
SYMBOL_DETECTION_PROMPT_V1 = """You are analyzing a tile cropped from a Piping & Instrumentation \
Diagram (P&ID). Detect every symbol (valve, instrument, flange, nozzle, safety device, or \
other P&ID symbol) visible in this image.

Respond with ONLY a JSON array of arrays, no other text, no explanation. Each inner array is
exactly: [x0, y0, x1, y1, confidence, "entity_type"]

Example: [[100, 200, 150, 260, 0.95, "valve"], [400, 50, 430, 90, 0.88, "instrument"]]

Coordinates are absolute pixel coordinates in this image (top-left origin, x right, y down),
NOT normalized. confidence is a float 0.0-1.0. If you see no symbols, respond with an empty
array: []"""

# v2 (engineered) — same rationale as Molmo's v2: real v1 results showed strong
# density-degradation (both models got worse on dense sheets) and mediocre precision (both
# models). Changes: (1) explicit exhaustive/systematic-scan instruction; (2) explicit
# symbol definition matching Gupta ground truth (rule 5 — class-agnostic icons only, not
# lines/text/dimensions); (3) tight-bbox instruction (loose boxes risk the match-point,
# the box center, landing outside the real symbol); (4) confidence calibration reminder.
SYMBOL_DETECTION_PROMPT_V2 = """You are an expert P&ID (Piping & Instrumentation Diagram) \
reviewer analyzing a cropped tile. This tile may contain many symbols — sometimes 50 or \
more. Scan the ENTIRE image systematically, region by region (top-left to bottom-right), \
and do not stop until you have checked every part of the image.

Count as a "symbol": valves, instruments (circles/bubbles), flanges, nozzles, safety \
devices, reducers, and other discrete P&ID equipment icons.
Do NOT count as a symbol: straight pipe/line segments, text labels alone, dimension \
arrows, or the tile border.

For each symbol you find, draw a TIGHT bounding box around just that symbol's icon (not \
surrounding text or connecting lines).

Respond with ONLY a JSON array of arrays, no other text, no explanation. Each inner array is
exactly: [x0, y0, x1, y1, confidence, "entity_type"]

Example: [[100, 200, 150, 260, 0.95, "valve"], [400, 50, 430, 90, 0.88, "instrument"]]

Coordinates are absolute pixel coordinates in this image (top-left origin, x right, y down),
NOT normalized. confidence is a float 0.0-1.0, calibrated to your actual certainty (not \
always high). If you see no symbols, respond with an empty array: []"""

# v3 (engineered) — two highest-leverage fixes over v2: (a) v2 forbade any text before the
# JSON ("ONLY a JSON array"), which blocks Claude from scanning/enumerating first — exactly
# when recall collapses on dense 50+ symbol tiles; v3 lets it scan region-by-region, then
# emits JSON in a fenced block that parse_claude extracts from the tail. (b) v2 never told
# Claude the tile's pixel size, so it had to guess the coordinate scale — v3 injects actual
# {W}x{H} (run_claude .format()s it per crop). Also keeps v2's symbol definition + tight-box
# + calibration guidance.
SYMBOL_DETECTION_PROMPT_V3 = """You are an expert P&ID reviewer analyzing one tile cropped \
from a larger Piping & Instrumentation Diagram. The image is {W}x{H} pixels.

First, briefly scan the tile region by region (top-left, top-right, center, bottom-left, \
bottom-right) and note what equipment symbols you see in each region. Dense tiles can \
contain 50+ symbols - keep going until every region is covered.

Count as a symbol: valves, instrument circles/bubbles, flanges, nozzles, safety devices, \
reducers, and other discrete equipment icons.
Not symbols: plain pipe/line segments, text labels alone, dimension arrows, borders.

Then output your final answer as a JSON array of arrays inside a ```json code fence. \
Each inner array is exactly: [x0, y0, x1, y1, confidence, "entity_type"] - tight boxes \
around just the icon, absolute pixel coordinates (top-left origin), confidence 0.0-1.0 \
calibrated to your actual certainty. If no symbols: []"""

SYMBOL_DETECTION_PROMPT = SYMBOL_DETECTION_PROMPT_V3  # active prompt used by run_claude() below

def run_claude(image, prompt=SYMBOL_DETECTION_PROMPT, max_retries=2):
    buf = BytesIO()
    image.save(buf, format="PNG")
    img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

    # v3 prompt contains {W}x{H} placeholders — fill with this crop's real pixel size.
    # (v1/v2 have no braces, so .format() is a harmless no-op for them.)
    filled_prompt = prompt.format(W=image.width, H=image.height)

    t0 = time.time()
    resp = claude_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=8000,  # v3 lets Claude scan/enumerate before the JSON — needs headroom for scan text + 50+ boxes
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": img_b64}},
                {"type": "text", "text": filled_prompt},
            ],
        }],
    )
    latency = time.time() - t0
    text = resp.content[0].text
    return text, latency

def parse_claude(text):
    # v3 emits scan text THEN a fenced JSON block — extract the last fenced array. Fall back
    # to treating the whole response as JSON (v1/v2 behaviour) if no fence is present.
    fenced = re.findall(r'```(?:json)?\s*(\[[\s\S]*?\])\s*```', text)
    if fenced:
        cleaned = fenced[-1].strip()
    else:
        cleaned = text.strip()
        cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned)
        cleaned = re.sub(r'\s*```$', '', cleaned)
    try:
        data = json.loads(cleaned)
    except json.JSONDecodeError as e:
        return None, f"JSONDecodeError: {e}"
    if not isinstance(data, list):
        return None, f"expected a JSON array, got {type(data).__name__}"
    detections = []
    for i, item in enumerate(data):
        if not (isinstance(item, list) and len(item) == 6):
            return None, f"item {i} malformed: {item}"
        x0, y0, x1, y1, confidence, entity_type = item
        detections.append({
            "bbox": [float(x0), float(y0), float(x1), float(y1)],
            "confidence": float(confidence),
            "entity_type": str(entity_type),
        })
    return detections, None

## 8d. Run over all 20 test sheets, score — same harness, same data as Molmo2

Requires the same local `/content/gupta_test/{sheets_test,labels_test}` copy from
section 1 above. If you haven't run that in this session, re-run section 1 first.

In [12]:
SHEETS_TEST_DIR = Path("/content/gupta_test/sheets_test")
LABELS_TEST_DIR = Path("/content/gupta_test/labels_test")
assert SHEETS_TEST_DIR.exists(), "Run section 1 (mount Drive + copy test sheets) first"

test_ids = sorted(p.stem for p in SHEETS_TEST_DIR.glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
print(f"Found {len(test_ids)} test sheets (expected 20)")

claude_sheet_results = []
claude_totals = {"n_matched": 0, "n_pred": 0, "n_gt": 0}
claude_parse_failures = 0
claude_total_tiles = 0

for sheet_id in test_ids:
    img_path = next(SHEETS_TEST_DIR.glob(f"{sheet_id}.*"))
    label_path = LABELS_TEST_DIR / f"{sheet_id}.txt"
    sheet_img = Image.open(img_path).convert("RGB")
    W, H = sheet_img.size
    gt_boxes = load_gt_boxes(label_path, W, H)

    tiles = compute_tile_grid(W, H)
    tile_preds, tile_origins = [], []
    for t in tiles:
        claude_total_tiles += 1
        crop = sheet_img.crop((t["x0"], t["y0"], t["x1"], t["y1"]))
        raw_text, _latency = run_claude(crop)
        dets, err = parse_claude(raw_text)
        if err:
            claude_parse_failures += 1
            dets = []
        tile_preds.append(dets)
        tile_origins.append((t["x0"], t["y0"]))

    sheet_preds, n_raw = stitch_and_dedup(tile_preds, tile_origins)
    result = precision_recall_f1(sheet_preds, gt_boxes)
    claude_sheet_results.append({"sheet_id": sheet_id, **result})
    for k in claude_totals:
        claude_totals[k] += result[k]
    print(f"{sheet_id:10s} tiles={len(tiles):2d} gt={result['n_gt']:3d} pred={result['n_pred']:3d} "
          f"matched={result['n_matched']:3d} precision={result['precision']:.2f} recall={result['recall']:.2f} f1={result['f1']:.2f}")

print(f"\n--- Overall Claude ({CLAUDE_MODEL}) — micro-averaged across all {len(test_ids)} test sheets, {claude_total_tiles} tiles ---")
cp = claude_totals["n_matched"] / claude_totals["n_pred"] if claude_totals["n_pred"] else 0.0
cr = claude_totals["n_matched"] / claude_totals["n_gt"] if claude_totals["n_gt"] else 0.0
cf1 = 2*cp*cr/(cp+cr) if (cp+cr) else 0.0
print(f"Precision: {cp:.3f} | Recall: {cr:.3f} | F1: {cf1:.3f}")
print(f"Parse failures: {claude_parse_failures}/{claude_total_tiles} tiles ({100*claude_parse_failures/claude_total_tiles:.1f}%)")
print(f"\n(Reference only per CLAUDE.md — ground truth is the pass bar, not this comparison.)")

Found 20 test sheets (expected 20)
0          tiles= 2 gt= 11 pred= 13 matched=  2 precision=0.15 recall=0.18 f1=0.17
103        tiles= 2 gt= 18 pred= 21 matched=  7 precision=0.33 recall=0.39 f1=0.36
11         tiles= 1 gt= 49 pred= 34 matched=  8 precision=0.24 recall=0.16 f1=0.19
124        tiles= 1 gt=  8 pred=  8 matched=  5 precision=0.62 recall=0.62 f1=0.62
129        tiles= 1 gt= 22 pred= 21 matched=  5 precision=0.24 recall=0.23 f1=0.23
136        tiles= 4 gt=  9 pred= 18 matched=  6 precision=0.33 recall=0.67 f1=0.44
145        tiles= 4 gt=  2 pred= 21 matched=  2 precision=0.10 recall=1.00 f1=0.17
148        tiles= 6 gt= 11 pred= 13 matched=  5 precision=0.38 recall=0.45 f1=0.42
15         tiles= 2 gt= 44 pred= 37 matched= 10 precision=0.27 recall=0.23 f1=0.25
151        tiles=12 gt=101 pred=113 matched= 44 precision=0.39 recall=0.44 f1=0.41
157        tiles= 6 gt= 46 pred= 50 matched=  9 precision=0.18 recall=0.20 f1=0.19
158        tiles= 8 gt= 60 pred= 49 matched= 25 prec